<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/03_search/search_error_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Search Error Analysis

## Objective
This notebook analyzes failure cases in semantic search results
to understand why incorrect rankings occur.

The goal is to identify common sources of retrieval errors
and build intuition about system limitations.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
query = "learn machine learning basics"

documents = [
    "Machine learning tutorials for beginners",
    "Introduction to deep learning",
    "Python programming fundamentals",
    "Natural language processing with transformers",
    "Football match highlights",
    "Top travel destinations"
]

In [ ]:
relevance = {
    "Machine learning tutorials for beginners": 1,
    "Introduction to deep learning": 1,
    "Python programming fundamentals": 0,
    "Natural language processing with transformers": 0,
    "Football match highlights": 0,
    "Top travel destinations": 0
}

In [7]:
query_emb = model.encode(query)
doc_embs = model.encode(documents)

scores = cosine_similarity([query_emb], doc_embs)[0]

df = pd.DataFrame({
    "Document": documents,
    "Similarity Score": np.round(scores, 4),
    "Relevant": [relevance[d] for d in documents]
}).sort_values(by="Similarity Score", ascending=False)

df

,Document,Similarity Score,Relevant
0,Machine learning tutorials for beginners,0.8806,1
1,Introduction to deep learning,0.5455,1
2,Python programming fundamentals,0.3898,0
3,Natural language processing with transformers,0.2065,0
4,Football match highlights,0.0823,0
5,Top travel destinations,0.0293,0


In [8]:
df["Error"] = df.apply(
    lambda row: "False Positive" if row["Similarity Score"] > 0.4 and row["Relevant"] == 0
    else ("Missed Relevant" if row["Similarity Score"] < 0.4 and row["Relevant"] == 1 else "Correct"),
    axis=1
)

df

,Document,Similarity Score,Relevant,Error
0,Machine learning tutorials for beginners,0.8806,1,Correct
1,Introduction to deep learning,0.5455,1,Correct
2,Python programming fundamentals,0.3898,0,Correct
3,Natural language processing with transformers,0.2065,0,Correct
4,Football match highlights,0.0823,0,Correct
5,Top travel destinations,0.0293,0,Correct


## Observations

- Some irrelevant documents may appear high in the ranking
- Some relevant documents may appear lower than expected
- Semantic similarity can confuse related topics

## Key Insight
Error analysis helps diagnose search weaknesses and guides
future improvements such as better embeddings or query expansion.